# Modelo RNN con metodo por Transecto y metodo General

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
RNN (Encoder-Decoder GRU) con KerasTuner.
No entrena si no hay datos de validación.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, RepeatVector, TimeDistributed, Reshape, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import keras_tuner as kt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")
MODELS_DIR = os.path.join(BASE_DIR, "models")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")  # para cargar CSV originales (necesario para estaciones)

DATA_DIR = WINDOWS_PARTITIONED_DIR
OUTPUT_DIR = os.path.join(MODELS_DIR, "rnn")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)

HP_UNITS = [64, 128]          # elimina 32
HP_DROPOUT = [0.2, 0.3]       # elimina 0.0
HP_LR = [1e-3, 1e-4]          # elimina 5e-4
HP_EPOCHS = 100
BATCH_SIZE = 32


def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan


def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }


def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def build_model(hp, input_shape, output_dim=72):
    if isinstance(hp, dict):
        units = hp['units']
        dropout = hp['dropout']
        lr = hp['lr']
    else:
        units = hp.Choice('units', HP_UNITS)
        dropout = hp.Choice('dropout', HP_DROPOUT)
        lr = hp.Choice('lr', HP_LR)
    encoder_inputs = Input(shape=input_shape, name='encoder_input')
    encoder = GRU(units, return_state=True, dropout=dropout, recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    _, state_h = encoder(encoder_inputs)
    decoder_repeated = RepeatVector(output_dim)(state_h)
    decoder_gru = GRU(units, return_sequences=True, dropout=dropout, recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    decoder_outputs = decoder_gru(decoder_repeated, initial_state=state_h)
    decoder_dense = TimeDistributed(Dense(1))(decoder_outputs)
    outputs = Reshape((output_dim,))(decoder_dense)
    model = Model(encoder_inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model


def get_station_idx_and_mapping(entity_name, transect=True):
    if not transect:
        return None, None
    csv_path = os.path.join(ENCODED_DIR, "dl", "by_transect", f"{entity_name}.csv")
    mapping_path = os.path.join(ENCODED_DIR, "dl", "by_transect", f"{entity_name}_mapping.json")
    if not os.path.exists(csv_path):
        return None, None
    df_cols = pd.read_csv(csv_path, nrows=0, index_col=0)
    feature_names = df_cols.columns.tolist()
    if 'Estacion' not in feature_names:
        return None, None
    idx_estacion = feature_names.index('Estacion')
    if os.path.exists(mapping_path):
        with open(mapping_path, 'r') as f:
            mapping_data = json.load(f)
        est_map = mapping_data.get('Estacion', {})
        mapping_estacion = {int(v): k for k, v in est_map.items()}
    else:
        mapping_estacion = None
    return idx_estacion, mapping_estacion


def compute_per_station_metrics_rnn(y_true, y_pred, X_test, idx_estacion, mapping_estacion, output_dir, entity_name):
    station_preds = {}
    for i in range(len(y_true)):
        station_code = int(round(X_test[i, 0, idx_estacion]))
        station_name = mapping_estacion.get(station_code, f"Unknown_{station_code}")
        station_preds.setdefault(station_name, {'true': [], 'pred': []})
        station_preds[station_name]['true'].append(y_true[i])
        station_preds[station_name]['pred'].append(y_pred[i])
    station_metrics = {}
    for station, data in station_preds.items():
        true_stack = np.vstack(data['true'])
        pred_stack = np.vstack(data['pred'])
        metrics = compute_metrics(true_stack.ravel(), pred_stack.ravel())
        station_metrics[station] = metrics
    if station_metrics:
        df = pd.DataFrame(station_metrics).T
        df.index.name = 'station'
        df.to_csv(os.path.join(output_dir, f"{entity_name}_per_station_metrics.csv"))
    return station_metrics


def train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                           scaler_y, entity_name, output_subdir,
                           idx_estacion=None, mapping_estacion=None):
    print(f"\n--- Entrenando RNN para {entity_name} ---")
    if len(X_val) == 0 or len(y_val) == 0:
        print(f"  Saltando {entity_name}: conjunto de validación vacío.")
        return None, None

    input_shape = (X_train.shape[1], X_train.shape[2])
    tuner = kt.RandomSearch(
        hypermodel=lambda hp: build_model(hp, input_shape),
        objective='val_loss',
        max_trials=len(HP_UNITS)*len(HP_DROPOUT)*len(HP_LR),
        executions_per_trial=1,
        directory=output_subdir,
        project_name='tuning',
        overwrite=True
    )
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    print("  Buscando hiperparámetros...")
    tuner.search(X_train, y_train, validation_data=(X_val, y_val),
                 epochs=HP_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, reduce_lr], verbose=1)
    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_params = {'units': best_hp.get('units'), 'dropout': best_hp.get('dropout'), 'lr': best_hp.get('lr')}
    print(f"  Mejores parámetros: {best_params}")

    model = build_model(best_params, input_shape)
    X_train_full = np.concatenate([X_train, X_val], axis=0)
    y_train_full = np.concatenate([y_train, y_val], axis=0)
    history = model.fit(X_train_full, y_train_full, validation_split=0.1, epochs=HP_EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                                   ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)], verbose=1)

    y_pred_scaled = model.predict(X_test, verbose=0)
    y_test_descaled = scaler_y.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    y_pred_descaled = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).reshape(y_pred_scaled.shape)
    test_metrics = compute_metrics(y_test_descaled.ravel(), y_pred_descaled.ravel())
    print(f"  Métricas test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}")

    if idx_estacion is not None and mapping_estacion is not None:
        station_metrics = compute_per_station_metrics_rnn(y_test_descaled, y_pred_descaled, X_test,
                                                          idx_estacion, mapping_estacion, output_subdir, entity_name)
    else:
        station_metrics = None

    model.save(os.path.join(output_subdir, "model.keras"))
    with open(os.path.join(output_subdir, "history.pkl"), 'wb') as f:
        pickle.dump(history.history, f)
    results = {'best_params': best_params, 'test_metrics': test_metrics, 'station_metrics': station_metrics,
               'n_train': len(X_train_full), 'n_test': len(X_test)}
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    horizons = [23, 47, 71]
    plot_predictions(y_test_descaled, y_pred_descaled, horizons,
                     os.path.join(output_subdir, "test_scatter.png"), f"RNN - {entity_name} - Test")
    np.save(os.path.join(output_subdir, "test_pred.npy"), y_pred_descaled)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test_descaled)
    return model, test_metrics


def process_by_transect():
    print("\n" + "="*50)
    print("PROCESANDO RNN POR TRANSECTO")
    dl_dir = os.path.join(DATA_DIR, "by_transect", "dl")
    if not os.path.exists(dl_dir):
        return
    entities = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    for entity in entities:
        entity_path = os.path.join(dl_dir, entity)
        X_train = np.load(os.path.join(entity_path, "train_X.npy"))
        y_train = np.load(os.path.join(entity_path, "train_y.npy"))
        X_val   = np.load(os.path.join(entity_path, "val_X.npy"))
        y_val   = np.load(os.path.join(entity_path, "val_y.npy"))
        X_test  = np.load(os.path.join(entity_path, "test_X.npy"))
        y_test  = np.load(os.path.join(entity_path, "test_y.npy"))
        with open(os.path.join(entity_path, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)
        idx_estacion, mapping_estacion = get_station_idx_and_mapping(entity, transect=True)
        out_subdir = os.path.join(OUTPUT_DIR, "by_transect", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                               scaler_y, entity, out_subdir, idx_estacion, mapping_estacion)


def process_global():
    print("\n" + "="*50)
    print("PROCESANDO RNN GLOBAL")
    dl_dir = os.path.join(DATA_DIR, "global", "dl")
    if not os.path.exists(dl_dir):
        return
    entities = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    for entity in entities:
        entity_path = os.path.join(dl_dir, entity)
        X_train = np.load(os.path.join(entity_path, "train_X.npy"))
        y_train = np.load(os.path.join(entity_path, "train_y.npy"))
        X_val   = np.load(os.path.join(entity_path, "val_X.npy"))
        y_val   = np.load(os.path.join(entity_path, "val_y.npy"))
        X_test  = np.load(os.path.join(entity_path, "test_X.npy"))
        y_test  = np.load(os.path.join(entity_path, "test_y.npy"))
        with open(os.path.join(entity_path, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)
        out_subdir = os.path.join(OUTPUT_DIR, "global", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                               scaler_y, entity, out_subdir, None, None)


def generate_summary():
    summary = []
    for t in ["by_transect", "global"]:
        dir_path = os.path.join(OUTPUT_DIR, t)
        if not os.path.exists(dir_path):
            continue
        for entity in os.listdir(dir_path):
            res_file = os.path.join(dir_path, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({'entity': entity, 'type': t, **data['test_metrics']})
    if summary:
        pd.DataFrame(summary).to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado.")


if __name__ == "__main__":
    print("RNN (GRU)")
    process_by_transect()
    process_global()
    generate_summary()

Trial 8 Complete [00h 14m 11s]
val_loss: 0.021832143887877464

Best val_loss So Far: 0.016665581613779068
Total elapsed time: 03h 00m 14s
  Mejores parámetros: {'units': 128, 'dropout': 0.2, 'lr': 0.001}


Epoch 1/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 63s 200ms/step - loss: 0.0314 - mae: 0.1387 - val_loss: 0.0223 - val_mae: 0.1177 - learning_rate: 0.0010
Epoch 2/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 47s 190ms/step - loss: 0.0250 - mae: 0.1264 - val_loss: 0.0218 - val_mae: 0.1151 - learning_rate: 0.0010
Epoch 3/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 47s 189ms/step - loss: 0.0239 - mae: 0.1239 - val_loss: 0.0232 - val_mae: 0.1172 - learning_rate: 0.0010
Epoch 4/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 47s 189ms/step - loss: 0.0233 - mae: 0.1221 - val_loss: 0.0232 - val_mae: 0.1172 - learning_rate: 0.0010
Epoch 5/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 50s 203ms/step - loss: 0.0227 - mae: 0.1209 - val_loss: 0.0233 - val_mae: 0.1176 - learning_rate: 0.0010
Epoch 6/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 51s 204ms/step - loss: 0.0191 - mae: 0.1079 - val_loss: 0.0191 - val_mae: 0.1054 - learning_rate: 0.0010
Epoch 7/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 47s 191ms/step - loss: 0.0162 - mae: 0.0975 - val_loss: 0.0201 - val_mae: 0.1069 - 